In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Flow of Data Analysis Project

1.   Agenda | Problem to Solve. Example - Maximize sales. It is different based on the domain
2.   Finding KPI's (Key Performance Indicators) : KPI's is someting which fulfills the problem. Basically understanding how the problem can be solved by asking certain questions. Ex - Maximize performacne of company. Now these can be done by different factors like maximize stock, etc.
3. Gathering Data
4. Data Exploration - Explore the data by removing irrelevant data
5. Data Cleaning (difficult part sometimes)
6. Data Analysis
7. Data Visualization
8. Summary of the analysis or the final result





In [ ]:
# cloning dataset from github
!git clone https://github.com/AshishJangra27/datasets.git

## problem statment
1.   Skill / location based job search
2. Salary Compensation



In [ ]:
df = pd.read_csv("/content/datasets/Job Postings/jobs.csv.zip")

In [ ]:
df.head()

## Data Analysis Steps


1.   Data Exploration
    - Explore the data and datatypes
    - remove unwanted columns
    - find null values
    - find duplicate values
    - explore each column
2.   Data Cleaning
    *   Dealing with null values
    *   dealing with duplicates
    *   Clean Columns
3. Analyze Data
  *   Questions for analysis. Answering questions by analyzing the data based on KPI's











### Data Exploration

In [ ]:
# basic information of dataset
df.info()

In [ ]:
# basic description of data (works only on numerical features)
df.describe()

In [ ]:
# finding null or missing values in the dataset
df.isnull().sum()

In [ ]:
# checking duplicate values
# based on job id or job link
df.duplicated(subset="job_id").sum()

In [ ]:
## removing unwanted columns
df.drop(columns=['posted_on','job_link','company_link'],axis = 1, inplace=True)

In [ ]:
df.columns

In [ ]:
len(df['experience'].unique())

In [ ]:
df['location'].value_counts().head(20)

In [ ]:
df['experience'].value_counts()

### Data Cleaning

#### 2.1 Dealing with null values (removing them)



In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(subset=['job_id','resposibilities','company'],inplace=True)
'''
1. Remvoed job id since they were somethign important and without that no analysis can be done
2. Removed responsibilites and company since their few of them which were null
'''

In [ ]:
df['experience'].mode()[0]

In [ ]:
# getting rid of null values (not the best way in this case)
df['experience'].fillna('5-10 Yrs',inplace=True)

In [ ]:
most_frequent_location = df['location'].mode()[0]
df['location'].fillna(most_frequent_location,inplace=True)

In [ ]:
df['rating'].isnull().sum()

In [ ]:
df['rating'].fillna(df['rating'].mean(),inplace=True)

In [ ]:
df['reviews'].fillna('0 Reviews',inplace=True)

#### 2.2 Dealing with duplicate values

In [ ]:
df.duplicated(subset='job_id').sum()

In [ ]:
df.drop_duplicates(subset='job_id',inplace=True)

#### 2.3 Convert data type of job id



In [ ]:
df['job_id'].head()

In [ ]:
df['job_id'] = df['job_id'].astype('int').astype('str')

#### 2.4 clean experience column

In [ ]:
df['experience'].unique()[:20]

In [ ]:
df['min_experience']=df['experience'].str.replace('Yrs','').str.split('-').str[0].str.strip().astype('int')

In [ ]:
df['max_experience']=df['experience'].str.replace('Yrs','').str.split('-').str[1].str.strip().astype('int')

#### 2.5 clean salary column

In [ ]:
df['clean_salary'] = df['salary'].str.replace(' PA.','').str.replace(',','').str.split('-').str[0].str.strip()

In [ ]:
df['clean_salary'] = df['salary'].str.replace(' PA.','').str.replace(',','').str.split('-').str[1].str.strip().str.split('(').str[0].str.strip()

In [ ]:
df['clean_salary'].unique()

In [ ]:
df = df[(df['clean_salary'] != '9.5 Cr and above') & (df['clean_salary'] != 'Less than 5000') & (df['clean_salary'] != 'Less than 50000') & (df['clean_salary'] != '')]

In [ ]:
values_to_be_replaced = ['MS/M.Sc(Science) MBA/PGDM M.A','MCA B.Tech/B.E.','B.Tech/B.E. BCA','MBBS','M.Tech MCA MS/M.Sc(Science) B.Tech/B.E.','Graduation Not Required','Any Postgraduate Any Graduate','MS/M.Sc(Science) M.A','MBA/PGDM','CA MBA/PGDM M.Com B.Com','Any Graduate']
df['clean_salary']=df['clean_salary'].replace(values_to_be_replaced, 'Not disclosed')

In [ ]:
df['min_salary'] = df['clean_salary'].str.strip()

In [ ]:
df['max_salary']=df['clean_salary'].fillna('Not disclosed')

#### 2.6 clean location column

In [ ]:
df['clean_location']=df['location'].str.replace('(All Areas)','').str.replace(',','').str.replace('/',' ').str.title().replace(r'\s+','-',regex=True).str.split('-')

#### 2.7 Clean reviews column

In [ ]:
df['reviews']=df['reviews'].str.split(' ').str[0].astype(int)

### Data Analysis

#### 3.1 Overall analysis

In [ ]:
print(f"Total number of unique jobs - {df['job_id'].nunique()}")
print(f"Total number of companies - {df['company'].nunique()}")

#### 3.2 Top 5 companies
- based on rating
- reviews
- salary
- no of job posting
- no of different locations



In [ ]:
# based on number of jobs
df['company'].value_counts().head()

In [ ]:
# based on rating
df.sort_values(by='rating',ascending=False).head()

In [ ]:
df.sort_values(by='reviews',ascending=False).head()

In [ ]:
# using group by clause
df.groupby('company')['reviews'].min().sort_values(ascending=False).head().plot(kind='barh')

#### 3.3 Companies Hiring for Data Analyst



In [ ]:
df[df['job_role'] == 'Data Analyst'].shape[0]

In [ ]:
df[df['job_role'] == 'Data Analyst'].head()

#### 3.4 Skills needed for almost all the jobs

In [ ]:
df['resposibilities'].str.lower().str.split(',').explode().value_counts().head(10)

In [ ]:
# Calculate the top 10 skills
top_skills = df['resposibilities'].str.lower().str.split(',').explode().value_counts().head(10)

# Plot a pie chart
plt.figure(figsize=(8, 8))
plt.pie(top_skills, labels=top_skills.index, autopct='%1.1f%%', startangle=140)
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.title('Top 10 Required Skills')
plt.show()

#### 3.5 Skills needed to get hired in HDFC bank

In [ ]:
df[df['company'].str.upper() == 'HDFC BANK']['resposibilities'].str.lower().str.split(',').explode().value_counts().head(10)

#### 3.6 Skills needed to become a data analyst

In [ ]:
df[df['job_role'].str.lower().str.contains("aws")]['job_role'].unique()

In [ ]:
df[df['job_role'].str.upper() == 'DATA SCIENTIST']['resposibilities'].str.lower().str.split(',').explode().value_counts().head().plot(kind='pie')